In [ ]:
!pip install torch transformers librosa datasets 
!pip install evaluate jiwer

In [ ]:
import torch
import librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from datasets import load_dataset

# Wav2Vec 추론 기본 코드

In [ ]:
model_name = 'kresnik/wav2vec2-large-xlsr-korean' # 모델 선정
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

file_path = 'sample_audio.wav'
speech, sr = librosa.load(file_path, sr=16000)

# 16kHz로 통일
input_values = processor(speech, return_tensors="pt", sampling_rate=16000).input_values

with torch.no_grad():
  logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)

print("Text: ", transcription[0])

In [ ]:
# kresnik/wav2vec2-large-xlsr-korean

processor = Wav2Vec2Processor.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")

model = Wav2Vec2ForCTC.from_pretrained("kresnik/wav2vec2-large-xlsr-korean").to('cuda')

ds = load_dataset("bingsu/zeroth-korean")

test_ds = ds['test']

def map_to_pred(batch):
    speech_samples = [item['array'] for item in batch['audio']]
    inputs = processor(speech_samples, sampling_rate=16000, return_tensors="pt", padding="longest")
    input_values = inputs.input_values.to("cuda")
    #attention_mask = inputs.attention_mask.to("cuda")
    
    with torch.no_grad():
        #logits = model(input_values, attention_mask=attention_mask).logits
        logits = model(input_values).logits
    
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)
    batch["transcription"] = transcription
    return batch

result = test_ds.map(map_to_pred, batched=True, batch_size=16)

import evaluate as eval
wer = eval.load("wer")
cer = eval.load("cer")
result_wer = wer.compute(references=result["text"], predictions=result["transcription"])
result_cer = cer.compute(references=result["text"], predictions=result["transcription"])
print(f'WER:{result_wer*100:.2f}%')
print(f'CER:{result_cer*100:.2f}%')


In [ ]:
# facebook/wav2vec2-large-960h

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from datasets import load_dataset
import torch

# load model and processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")
    
# load dummy dataset and read soundfiles
ds = load_dataset("patrickvonplaten/librispeech_asr_dummy", "clean", split="validation")

# tokenize
input_values = processor(ds[0]["audio"]["array"], return_tensors="pt", padding="longest").input_values  # Batch size 1

# retrieve logits
logits = model(input_values).logits

# take argmax and decode
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)


In [ ]:
# wav2vec2 + forced alignment


In [ ]:
# GOP